In [2]:
# Train at 8/255
eps_train = 8 / 255

model_linf_8 = load_base_model()

model_linf_8 = adv_train(
    model_linf_8,
    single_norm_loss,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    norm="linf",
    eps=eps_train,
    steps=10
)

print("Adversarial training completed!")


NameError: name 'load_base_model' is not defined

In [ ]:
# Save 8/255 checkpoint
torch.save(
    model_linf_8.state_dict(),
    "CS521_HW3_linf_eps8.pt"
)

print("Model saved successfully!")


## Accuracy tests

Clean accuracy uses original images. PGD accuracy counts images that remain correct after the attacks.

In [ ]:
# Clean accuracy
def test_clean_accuracy(model, x, y, batch_size=100):
    model.eval()
    correct = 0

    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            x_batch = x[i:i + batch_size]
            y_batch = y[i:i + batch_size]

            predictions = model(x_batch).argmax(dim=1)
            correct += (predictions == y_batch).sum().item()

    return 100 * correct / len(y)


original_model = load_base_model()

original_acc = test_clean_accuracy(
    original_model, x_test, y_test
)

trained_acc = test_clean_accuracy(
    model_linf_8, x_test, y_test
)

print(f"Original model standard accuracy: {original_acc:.2f}%")
print(f"Trained model standard accuracy: {trained_acc:.2f}%")


In [ ]:
# PGD test function
def test_pgd_accuracy(
    model, x, y, eps=8/255,
    steps=50, restarts=2, batch_size=100
):
    model.eval()

    alpha = 2.5 * eps / steps
    correct = 0

    torch.manual_seed(0)

    for i in range(0, len(x), batch_size):
        x_batch = x[i:i + batch_size]
        y_batch = y[i:i + batch_size]

        with torch.no_grad():
            robust = (
                model(x_batch).argmax(1) == y_batch
            )

        for _ in range(restarts):
            x_adv = pgd_linf(
                model, x_batch, y_batch,
                eps, alpha, steps
            )

            with torch.no_grad():
                robust &= (
                    model(x_adv).argmax(1) == y_batch
                )

        correct += robust.sum().item()

    return 100 * correct / len(y)


In [3]:
# Compare PGD accuracy
original_robust = test_pgd_accuracy(
    original_model, x_test, y_test
)

trained_robust = test_pgd_accuracy(
    model_linf_8, x_test, y_test
)

print("Original model:")
print(f"  Standard accuracy: {original_acc:.2f}%")
print(f"  PGD robust accuracy: {original_robust:.2f}%")

print("\nAdversarially trained model:")
print(f"  Standard accuracy: {trained_acc:.2f}%")
print(f"  PGD robust accuracy: {trained_robust:.2f}%")

print(
    "\nRobust accuracy difference:",
    f"{trained_robust - original_robust:+.2f} percentage points"
)


NameError: name 'test_pgd_accuracy' is not defined

In [ ]:
# Test attack epsilon values
epsilon_values = [2/255, 4/255, 8/255, 12/255]

pgd_results = []

for eps in epsilon_values:

    print(f"\nTesting epsilon = {eps * 255:.0f}/255")

    original_robust = test_pgd_accuracy(
        original_model,
        x_test,
        y_test,
        eps=eps,
        steps=50,
        restarts=2
    )

    trained_robust = test_pgd_accuracy(
        model_linf_8,
        x_test,
        y_test,
        eps=eps,
        steps=50,
        restarts=2
    )

    pgd_results.append({
        "epsilon": f"{eps * 255:.0f}/255",
        "original": original_robust,
        "trained": trained_robust
    })

    print(f"Original model: {original_robust:.2f}%")
    print(f"Trained model:  {trained_robust:.2f}%")


In [ ]:
# PGD results table
import pandas as pd

pgd_df = pd.DataFrame(pgd_results)

pgd_df["difference"] = (
    pgd_df["trained"] - pgd_df["original"]
)

print(pgd_df.to_string(index=False))


## Part (b): PGD and FGSM

FGSM uses one gradient step. PGD uses 50 steps and two random starts for evaluation.

In [ ]:
# FGSM attack
def fgsm_attack(model, x, y, eps):

    delta = torch.zeros_like(x)

    grad = input_grad(model, x, delta, y)

    delta = eps * grad.sign()

    delta = clip_box(x, delta)

    return (x + delta).detach()


In [ ]:
# FGSM test function
def test_fgsm_accuracy(model, x, y, eps, batch_size=100):

    model.eval()
    correct = 0

    for i in range(0, len(x), batch_size):

        x_batch = x[i:i + batch_size]
        y_batch = y[i:i + batch_size]

        x_adv = fgsm_attack(
            model, x_batch, y_batch, eps
        )

        with torch.no_grad():
            clean_correct = (
                model(x_batch).argmax(1) == y_batch
            )

            adv_correct = (
                model(x_adv).argmax(1) == y_batch
            )

            correct += (
                clean_correct & adv_correct
            ).sum().item()

    return 100 * correct / len(y)


In [ ]:
# FGSM results
epsilon_values = [2/255, 4/255, 8/255, 12/255]

fgsm_results = []

for eps in epsilon_values:

    original_fgsm = test_fgsm_accuracy(
        original_model, x_test, y_test, eps
    )

    trained_fgsm = test_fgsm_accuracy(
        model_linf_8, x_test, y_test, eps
    )

    fgsm_results.append({
        "epsilon": f"{eps * 255:.0f}/255",
        "original": original_fgsm,
        "trained": trained_fgsm
    })

    print(f"\nFGSM epsilon = {eps * 255:.0f}/255")
    print(f"Original model: {original_fgsm:.2f}%")
    print(f"Trained model:  {trained_fgsm:.2f}%")


In [4]:
# Train at 4/255
eps_train_4 = 4 / 255

model_linf_4 = load_base_model()

model_linf_4 = adv_train(
    model_linf_4,
    single_norm_loss,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    norm="linf",
    eps=eps_train_4,
    steps=10
)

torch.save(
    model_linf_4.state_dict(),
    "CS521_HW3_linf_eps4.pt"
)

print("Training with epsilon = 4/255 completed!")


NameError: name 'load_base_model' is not defined

In [ ]:
# Test 4/255 model
clean_acc_4 = test_clean_accuracy(
    model_linf_4,
    x_test,
    y_test
)

robust_acc_4 = test_pgd_accuracy(
    model_linf_4,
    x_test,
    y_test,
    eps=8/255,
    steps=50,
    restarts=2
)

print("Model trained with epsilon = 4/255")
print(f"Standard accuracy: {clean_acc_4:.2f}%")
print(f"PGD robust accuracy at 8/255: {robust_acc_4:.2f}%")


In [ ]:
# Compare models
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Original pretrained",
        "Trained eps = 4/255",
        "Trained eps = 8/255"
    ],
    "Standard Accuracy (%)": [
        original_acc,
        clean_acc_4,
        trained_acc
    ],
    "PGD Robust Accuracy (%)": [
        48.30,
        robust_acc_4,
        44.80
    ]
})

print(comparison.to_string(index=False))


In [ ]:
# Train at 2/255
eps_train_2 = 2 / 255

model_linf_2 = load_base_model()

model_linf_2 = adv_train(
    model_linf_2,
    single_norm_loss,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    norm="linf",
    eps=eps_train_2,
    steps=10
)

torch.save(
    model_linf_2.state_dict(),
    "CS521_HW3_linf_eps2.pt"
)

print("Training with epsilon = 2/255 completed!")


In [ ]:
# Test 2/255 model
clean_acc_2 = test_clean_accuracy(
    model_linf_2,
    x_test,
    y_test
)

robust_acc_2 = test_pgd_accuracy(
    model_linf_2,
    x_test,
    y_test,
    eps=8/255,
    steps=50,
    restarts=2
)

print("Model trained with epsilon = 2/255")
print(f"Standard accuracy: {clean_acc_2:.2f}%")
print(f"PGD robust accuracy: {robust_acc_2:.2f}%")


In [ ]:
# Final training comparison
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Original pretrained",
        "Trained eps = 2/255",
        "Trained eps = 4/255",
        "Trained eps = 8/255"
    ],
    "Standard Accuracy (%)": [
        original_acc,
        clean_acc_2,
        clean_acc_4,
        trained_acc
    ],
    "PGD Robust Accuracy (%)": [
        48.30,
        robust_acc_2,
        robust_acc_4,
        44.80
    ]
})

print(comparison.to_string(index=False))


In [ ]:
# PGD and FGSM results
models = {
    "Original": original_model,
    "Train 2/255": model_linf_2,
    "Train 4/255": model_linf_4,
    "Train 8/255": model_linf_8
}

attack_epsilons = [2/255, 4/255, 8/255, 12/255]

results = []

for name, current_model in models.items():

    for eps in attack_epsilons:

        pgd_acc = test_pgd_accuracy(
            current_model,
            x_test,
            y_test,
            eps=eps,
            steps=50,
            restarts=2
        )

        fgsm_acc = test_fgsm_accuracy(
            current_model,
            x_test,
            y_test,
            eps=eps
        )

        results.append({
            "Model": name,
            "Attack epsilon": f"{eps * 255:.0f}/255",
            "PGD accuracy": pgd_acc,
            "FGSM accuracy": fgsm_acc
        })

import pandas as pd

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))


## Results from the recorded Colab run

These numbers are from the original run. Editing the notebook does not recalculate them.

| Model | Clean accuracy | PGD accuracy (attack ε = 8/255) |
|---|---:|---:|
| Original pretrained model | 78.70% | 48.30% |
| Trained at 2/255 | 85.20% | 27.60% |
| Trained at 4/255 | 83.20% | 36.30% |
| Trained at 8/255 | 78.60% | 44.80% |

**Part (a):** Among the fine-tuned models, larger training epsilon gave better PGD accuracy at attack epsilon 8/255, but lower clean accuracy.

**Part (b):** Accuracy fell as attack epsilon increased. PGD gave lower accuracy than FGSM for the tested settings. The original pretrained model was already adversarially trained, so this is not a comparison against ordinary training.

PGD evaluation used 50 steps and 2 restarts on 1,000 test images. These are measured attack accuracies, not formal robustness guarantees.
